In [ ]:
from __future__ import annotations

import re
from typing import Any

import mlflow
import mlflow.onnx
import numpy as np
import onnxruntime as ort
import pandas as pd
from mlflow.models.signature import infer_signature
from skl2onnx import to_onnx
from skl2onnx.common.data_types import FloatTensorType
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 42
TARGET_COLUMN = "label"
MAX_TRAINING_ROWS_DEFAULT = 200_000
CV_FOLDS_DEFAULT = 5
TEST_SIZE = 0.2
SVC_C = 1.0
SVC_KERNEL = "rbf"
TARGET_OPSET = 17


# Job parameters are passed in through notebook task base_parameters.
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "flip_flopper")
dbutils.widgets.text("table_name", "generated_data")
dbutils.widgets.text("experiment_name", "flip_flopper_classifier")
dbutils.widgets.text("registered_model_suffix", "flip_flopper_classifier")
dbutils.widgets.text("max_training_rows", str(MAX_TRAINING_ROWS_DEFAULT))
dbutils.widgets.text("cv_folds", str(CV_FOLDS_DEFAULT))


config = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "table_name": dbutils.widgets.get("table_name").strip(),
    "experiment_name": dbutils.widgets.get("experiment_name").strip(),
    "registered_model_suffix": dbutils.widgets.get("registered_model_suffix").strip(),
    "max_training_rows": dbutils.widgets.get("max_training_rows").strip(),
    "cv_folds": dbutils.widgets.get("cv_folds").strip(),
}


def quote_identifier(value: str) -> str:
    escaped = value.replace("`", "``")
    return f"`{escaped}`"


def qualified_table_name(catalog: str, schema: str, table_name: str) -> str:
    return ".".join(quote_identifier(part) for part in (catalog, schema, table_name))


def normalize_model_name(raw_value: str) -> str:
    cleaned = re.sub(r"[^0-9A-Za-z_]", "_", raw_value.strip().lower())
    cleaned = re.sub(r"_+", "_", cleaned).strip("_")
    if not cleaned:
        raise ValueError(
            "registered_model_suffix must contain at least one valid character"
        )
    return cleaned


def resolve_experiment_path(raw_value: str) -> str:
    return raw_value if raw_value.startswith("/") else f"/Shared/{raw_value}"


def load_training_frame(source_table: str, max_training_rows: int) -> pd.DataFrame:
    spark_df = spark.table(source_table)
    row_count = spark_df.count()

    if row_count > max_training_rows:
        sampling_fraction = min(1.0, (max_training_rows / row_count) * 1.2)
        spark_df = spark_df.sample(False, sampling_fraction, seed=RANDOM_STATE).limit(
            max_training_rows
        )
        print(
            f"Sampled approximately {max_training_rows:,} rows from {row_count:,} available rows"
        )
    else:
        print(f"Using all {row_count:,} rows from {source_table}")

    frame = spark_df.toPandas()
    if TARGET_COLUMN not in frame.columns:
        raise ValueError(f"Expected target column '{TARGET_COLUMN}' in {source_table}")

    return frame


def build_training_pipeline(feature_columns: list[str]) -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                feature_columns,
            )
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    classifier = SVC(
        kernel=SVC_KERNEL,
        C=SVC_C,
        probability=True,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )

    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("classifier", classifier),
        ]
    )


def summarize_cv_metrics(cv_results: dict[str, np.ndarray]) -> dict[str, float]:
    summary: dict[str, float] = {}
    for metric_name in ("accuracy", "f1", "roc_auc"):
        scores = cv_results[f"test_{metric_name}"]
        summary[f"cv_{metric_name}_mean"] = float(np.mean(scores))
        summary[f"cv_{metric_name}_std"] = float(np.std(scores))
    return summary


def safe_roc_auc_score(labels: pd.Series, probabilities: np.ndarray) -> float:
    unique_labels = np.unique(labels.to_numpy())
    if len(unique_labels) < 2:
        return float("nan")
    return float(roc_auc_score(labels, probabilities))


def evaluate_holdout(
    model: Pipeline, features: pd.DataFrame, labels: pd.Series
) -> dict[str, float]:
    predictions = model.predict(features)
    probabilities = model.predict_proba(features)[:, 1]
    return {
        "test_accuracy": float(accuracy_score(labels, predictions)),
        "test_f1": float(f1_score(labels, predictions)),
        "test_roc_auc": safe_roc_auc_score(labels, probabilities),
    }


def convert_pipeline_to_onnx(model: Pipeline, feature_columns: list[str]) -> Any:
    initial_types = [(column, FloatTensorType([None, 1])) for column in feature_columns]
    return to_onnx(
        model,
        initial_types=initial_types,
        target_opset=TARGET_OPSET,
        options={id(model.named_steps["classifier"]): {"zipmap": False}},
    )


def run_onnx_proba(
    onnx_model: Any, validation_frame: pd.DataFrame, feature_columns: list[str]
) -> np.ndarray:
    onnx_inputs = {
        column: validation_frame[column].to_numpy(dtype=np.float32).reshape((-1, 1))
        for column in feature_columns
    }
    session = ort.InferenceSession(
        onnx_model.SerializeToString(), providers=["CPUExecutionProvider"]
    )
    outputs = session.run(None, onnx_inputs)

    probability_output = next(
        (
            output
            for output in outputs
            if getattr(output, "ndim", 0) == 2
            and output.shape[0] == len(validation_frame)
            and output.shape[1] >= 2
        ),
        None,
    )
    if probability_output is None:
        raise ValueError("Unable to find probability output from ONNX runtime session")

    return probability_output


def validate_onnx_probabilities(
    model: Pipeline,
    onnx_model: Any,
    validation_frame: pd.DataFrame,
    feature_columns: list[str],
) -> tuple[float, np.ndarray]:
    expected = model.predict_proba(validation_frame)[:, 1]
    proba_2d = run_onnx_proba(onnx_model, validation_frame, feature_columns)
    onnx_positive = proba_2d[:, 1]
    max_abs_diff = float(np.max(np.abs(expected - onnx_positive)))
    return max_abs_diff, proba_2d


def build_onnx_input_example(
    validation_frame: pd.DataFrame, feature_columns: list[str]
) -> dict[str, np.ndarray]:
    return {
        column: validation_frame[[column]].to_numpy(dtype=np.float32)
        for column in feature_columns
    }


source_table = qualified_table_name(
    config["catalog"], config["schema"], config["table_name"]
)
experiment_path = resolve_experiment_path(config["experiment_name"])
registered_model_name = f"{config['catalog']}.{config['schema']}.{normalize_model_name(config['registered_model_suffix'])}_b"

training_frame = load_training_frame(
    source_table,
    max_training_rows=int(config["max_training_rows"]),
)
training_frame[TARGET_COLUMN] = pd.to_numeric(
    training_frame[TARGET_COLUMN], errors="raise"
).astype("int64")

feature_columns = sorted(
    column for column in training_frame.columns if column != TARGET_COLUMN
)
if not feature_columns:
    raise ValueError("Training data must contain at least one feature column")

features = (
    training_frame[feature_columns]
    .apply(pd.to_numeric, errors="coerce")
    .astype("float32")
)
labels = training_frame[TARGET_COLUMN]

x_train, x_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=TEST_SIZE,
    stratify=labels,
    random_state=RANDOM_STATE,
)

training_pipeline = build_training_pipeline(feature_columns)

requested_cv_folds = int(config["cv_folds"])
min_class_count = int(y_train.value_counts().min())
if min_class_count < 2:
    raise ValueError(
        "Need at least 2 samples in each class after train/test split for evaluation"
    )

effective_cv_folds = min(requested_cv_folds, min_class_count)
if effective_cv_folds < 2:
    raise ValueError(
        f"cv_folds must be >= 2 (requested {requested_cv_folds}, effective {effective_cv_folds})"
    )
if effective_cv_folds != requested_cv_folds:
    print(
        f"Adjusted cv_folds from {requested_cv_folds} to {effective_cv_folds} based on class balance"
    )

cv = StratifiedKFold(
    n_splits=effective_cv_folds, shuffle=True, random_state=RANDOM_STATE
)
cv_results = cross_validate(
    training_pipeline,
    x_train,
    y_train,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "f1": "f1",
        "roc_auc": "roc_auc",
    },
    n_jobs=1,
    return_train_score=False,
)

training_pipeline.fit(x_train, y_train)
fitted_svc = training_pipeline.named_steps["classifier"]
holdout_metrics = evaluate_holdout(training_pipeline, x_test, y_test)
cross_validation_metrics = summarize_cv_metrics(cv_results)

onnx_model = convert_pipeline_to_onnx(training_pipeline, feature_columns)
validation_batch = x_test.head(min(128, len(x_test))).copy()
onnx_input_example = build_onnx_input_example(validation_batch, feature_columns)

onnx_probability_max_abs_diff, onnx_proba_2d = validate_onnx_probabilities(
    training_pipeline, onnx_model, validation_batch, feature_columns
)

signature = infer_signature(validation_batch, onnx_proba_2d)

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(experiment_path)

with mlflow.start_run(run_name="train_classifier_b_svc") as run:
    mlflow.log_params(
        {
            "algorithm": "svc_rbf_classifier",
            "catalog": config["catalog"],
            "schema": config["schema"],
            "source_table": f"{config['catalog']}.{config['schema']}.{config['table_name']}",
            "registered_model_name": registered_model_name,
            "feature_count": len(feature_columns),
            "training_row_count": len(x_train),
            "test_row_count": len(x_test),
            "max_training_rows": int(config["max_training_rows"]),
            "cv_folds_requested": requested_cv_folds,
            "cv_folds_effective": effective_cv_folds,
            "svc_kernel": SVC_KERNEL,
            "svc_C": str(fitted_svc.C),
            "svc_gamma": str(fitted_svc.gamma_),
            "svc_probability": str(fitted_svc.probability),
        }
    )
    mlflow.log_metrics(cross_validation_metrics)
    mlflow.log_metrics(holdout_metrics)
    mlflow.log_metric("onnx_probability_max_abs_diff", onnx_probability_max_abs_diff)
    mlflow.log_table(pd.DataFrame(cv_results), artifact_file="cv_results.json")
    mlflow.log_text("\n".join(feature_columns), artifact_file="feature_columns.txt")

    model_info = mlflow.onnx.log_model(
        onnx_model=onnx_model,
        name="model",
        input_example=onnx_input_example,
        signature=signature,
        registered_model_name=registered_model_name,
        metadata={
            "algorithm": "svc_rbf_classifier",
            "source_table": f"{config['catalog']}.{config['schema']}.{config['table_name']}",
            "notebook": "train_classifier_b",
        },
    )

print(f"Experiment path: {experiment_path}")
print(f"Registered model: {registered_model_name}")
print(f"Model URI: {model_info.model_uri}")
print(f"Cross-validation metrics: {cross_validation_metrics}")
print(f"Holdout metrics: {holdout_metrics}")
print(f"ONNX probability max abs diff: {onnx_probability_max_abs_diff:.8f}")
